In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
#!pip install transformers accelerate langchain_huggingface "headroom-ai[langchain]" langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 12.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 82.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 61.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 82.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
  Attempting u

In [3]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers import pipeline


In [9]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from headroom import HeadroomConfig, HeadroomMode
from headroom.integrations import HeadroomChatModel

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    max_new_tokens=512,
)
base_llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))

#headroom_config = HeadroomConfig(default_mode=HeadroomMode.OPTIMIZE)
#optimized_llm = HeadroomChatModel(base_llm, config=headroom_config)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [15]:
import os

from langchain_core.messages import HumanMessage

In [14]:
from headroom import HeadroomConfig, HeadroomMode
from headroom.integrations import HeadroomChatModel

config = HeadroomConfig(
    default_mode=HeadroomMode.OPTIMIZE
)

optimized_llm = HeadroomChatModel(
    base_llm,
    config=config,
    mode=HeadroomMode.OPTIMIZE,
)

In [24]:
NOISY_TOOL_OUTPUT = """
[Tool result: user_database_search]
{"results": [
  {"id": "usr_483920", "email": "user0@example.com", "name": "User 0 Smith", "department": "Engineering", "status": "active", "role": "admin"},
  {"id": "usr_192837", "email": "user1@example.com", "name": "User 1 Johnson", "department": "Sales", "status": "inactive", "role": "user"},
  {"id": "usr_837465", "email": "user2@example.com", "name": "User 2 Williams", "department": "Support", "status": "pending", "role": "viewer"}
  /* ... imagine ~100 more rows like this in a real tool call ... */
], "total": 103, "query": "active engineers"}
"""

messages = [
    HumanMessage(
        content=(
            "Here is a database query result. Just tell me how many "
            f"users are in the Engineering department.\n\n{NOISY_TOOL_OUTPUT}"
        )
    )
]


In [26]:
print("=" * 60)
print("BASELINE (no Headroom)")
print("=" * 60)
baseline_response = base_llm.invoke(messages)
print(baseline_response.content)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASELINE (no Headroom)
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Here is a database query result. Just tell me how many users are in the Engineering department.


[Tool result: user_database_search]
{"results": [
  {"id": "usr_483920", "email": "user0@example.com", "name": "User 0 Smith", "department": "Engineering", "status": "active", "role": "admin"},
  {"id": "usr_192837", "email": "user1@example.com", "name": "User 1 Johnson", "department": "Sales", "status": "inactive", "role": "user"},
  {"id": "usr_837465", "email": "user2@example.com", "name": "User 2 Williams", "department": "Support", "status": "pending", "role": "viewer"}
  /* ... imagine ~100 more rows like this in a real tool call ... */
], "total": 103, "query": "active engineers"}
<|im_end|>
<|im_start|>assistant
Based on the provided query result, there is 1 user in the Engineering department. The query specifically filtered for "active engineers,

In [25]:
print("\n" + "=" * 60)
print("HEADROOM-WRAPPED")
print("=" * 60)
optimized_response = optimized_llm.invoke(messages)
print(optimized_response.content)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



HEADROOM-WRAPPED

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Here is a database query result. Just tell me how many users are in the Engineering department.


[Tool result: user_database_search]
{"results": [
  {"id": "usr_483920", "email": "user0@example.com", "name": "User 0 Smith", "department": "Engineering", "status": "active", "role": "admin"},
  {"id": "usr_192837", "email": "user1@example.com", "name": "User 1 Johnson", "department": "Sales", "status": "inactive", "role": "user"},
  {"id": "usr_837465", "email": "user2@example.com", "name": "User 2 Williams", "department": "Support", "status": "pending", "role": "viewer"}
  /* ... imagine ~100 more rows like this in a real tool call ... */
], "total": 103, "query": "active engineers"}
<|im_end|>
<|im_start|>assistant
Based on the provided query re

In [36]:
# HeadroomChatModel tracks real optimization metrics per call.
print("\nSavings summary:", optimized_llm.get_savings_summary())
print("Total tokens saved so far:", optimized_llm.total_tokens_saved)



Savings summary: {'total_requests': 1, 'total_tokens_saved': 0, 'average_savings_percent': 0.0, 'total_tokens_before': 13, 'total_tokens_after': 13}
Total tokens saved so far: 0


In [32]:
optimized_llm = HeadroomChatModel(base_llm)

    # Use normally - Headroom automatically optimizes context
response = optimized_llm.invoke("Capital of France?")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



In [33]:
response.content

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nCapital of France?<|im_end|>\n<|im_start|>assistant\nThe capital of France is Paris.'

In [35]:
base_llm.invoke("Capital of France?").content

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nCapital of France?<|im_end|>\n<|im_start|>assistant\nThe capital of France is Paris.'

In [39]:
from langchain_openai import ChatOpenAI
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from headroom.integrations import HeadroomChatModel

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    max_new_tokens=512,
)
base_llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
llm = HeadroomChatModel(base_llm)

# Use exactly like before
response = llm.invoke("Hello!")

# Check savings
print(llm.get_stats())
# {'tokens_saved': 12500, 'savings_percent': 45.2, 'requests': 50}

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



AttributeError: 'HeadroomChatModel' object has no attribute 'get_stats'

In [38]:
import headroom
print(headroom.__version__)

0.37.0


In [46]:
print("Total tokens saved:", llm.total_tokens_saved)

print("\nMetrics history:")
for metric in llm.metrics_history:
    print(metric)

print("\nSavings summary:")
print(llm.get_savings_summary())

Total tokens saved: 0

Metrics history:
OptimizationMetrics(request_id='5c8f3f89-1667-48e6-95d0-e2eb281bd058', timestamp=datetime.datetime(2026, 9, 1, 8, 33, 38, 459244), tokens_before=10, tokens_after=10, tokens_saved=0, savings_percent=0.0, transforms_applied=['router:protected:user_message'], model='Qwen/Qwen2.5-7B-Instruct')
OptimizationMetrics(request_id='ae33a95c-5505-4851-aba4-c1d7cd2ca23b', timestamp=datetime.datetime(2026, 9, 1, 8, 38, 55, 179710), tokens_before=57, tokens_after=57, tokens_saved=0, savings_percent=0.0, transforms_applied=['router:protected:user_message'], model='Qwen/Qwen2.5-7B-Instruct')
OptimizationMetrics(request_id='d1b9eadb-de30-4a9a-aaef-9155fd26a996', timestamp=datetime.datetime(2026, 9, 1, 8, 41, 42, 433926), tokens_before=10, tokens_after=10, tokens_saved=0, savings_percent=0.0, transforms_applied=['router:protected:user_message'], model='Qwen/Qwen2.5-7B-Instruct')

Savings summary:
{'total_requests': 3, 'total_tokens_saved': 0, 'average_savings_perce

In [47]:
response = llm.invoke("Hello!")

metric = llm.metrics_history[-1]

print("Provider:", type(llm._provider))
print("Model:", metric.model)
print("Before:", metric.tokens_before)
print("After:", metric.tokens_after)
print("Saved:", metric.tokens_saved)
print("Savings %:", metric.savings_percent)
print("Transforms:", metric.transforms_applied)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Provider: <class 'headroom.providers.openai.OpenAIProvider'>
Model: Qwen/Qwen2.5-7B-Instruct
Before: 10
After: 10
Saved: 0
Savings %: 0.0
Transforms: ['router:protected:user_message']


In [15]:
long_context = """
The following is a large collection of information about a software project.

Project name: Travel Route Optimization System.

The system is designed to find optimal routes between cities using graph algorithms.
It uses Python, FastAPI, igraph, Docker, and AWS.

The API accepts a source city and destination city and returns the optimal route,
total travel time, and intermediate cities.

The graph contains thousands of cities and many thousands of edges.

The system uses Dijkstra's shortest path algorithm to calculate optimal routes.

The API has a Swagger interface for testing endpoints.

The application is containerized using Docker.

The application can be deployed to AWS ECS.

The graph data is stored in CSV files before being converted into an igraph graph.

The graph contains nodes representing cities.

Each node has a city name, population, latitude, and longitude.

Edges represent connections between cities.

Each edge contains a travel time.

The system also calculates degree centrality.

The system calculates betweenness centrality.

The system identifies connected components.

The system performs community detection.

The API uses Pydantic models for request validation.

The API uses FastAPI for serving HTTP requests.

The API uses Uvicorn as the ASGI server.

The project uses logging for debugging and observability.

The application has unit tests.

The application has CI/CD through GitHub Actions.

The application can be deployed using Docker images.

""" * 5

response = llm.invoke(long_context)

metric = llm.metrics_history[-1]

print(f"Before:    {metric.tokens_before}")
print(f"After:     {metric.tokens_after}")
print(f"Saved:     {metric.tokens_saved}")
print(f"Savings:   {metric.savings_percent:.2f}%")
print(f"Transforms: {metric.transforms_applied}")



Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Before:    1869
After:     1869
Saved:     0
Savings:   0.00%
Transforms: ['router:protected:user_message']


In [13]:
print(llm.headroom_config)

HeadroomConfig(store_url='sqlite:///headroom.db', default_mode=<HeadroomMode.AUDIT: 'audit'>, model_context_limits={}, smart_crusher=SmartCrusherConfig(enabled=True, min_items_to_analyze=5, min_tokens_to_crush=200, variance_threshold=2.0, uniqueness_threshold=0.1, similarity_threshold=0.8, max_items_after_crush=15, preserve_change_points=True, factor_out_constants=False, include_summaries=False, use_feedback_hints=True, toin_confidence_threshold=0.3, relevance=RelevanceScorerConfig(tier='hybrid', bm25_k1=1.5, bm25_b=0.75, embedding_model='all-MiniLM-L6-v2', hybrid_alpha=0.5, adaptive_alpha=True, relevance_threshold=0.25), anchor=AnchorConfig(anchor_budget_pct=0.25, min_anchor_slots=3, max_anchor_slots=12, default_front_weight=0.5, default_back_weight=0.4, default_middle_weight=0.1, search_front_weight=0.75, search_back_weight=0.15, logs_front_weight=0.15, logs_back_weight=0.75, recency_keywords=('latest', 'recent', 'last', 'newest', 'current', 'now'), historical_keywords=('first', 'old